# Regularization: Ridge ($L_2$) and Lasso ($L_1$) Mechanics
Accompanies **Unit 3.2: Regularization Mechanics (L1, L2, and ElasticNet)**.

When features are correlated, $\mathbf{X}^T \mathbf{X}$ is ill-conditioned with near-zero eigenvalues, causing unconstrained OLS variance to explode.
- **Ridge ($L_2$):** Minimizes $\|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 + \alpha \|\mathbf{w}\|_2^2$. Shifts eigenvalues up by $\alpha$, shrinking weights smoothly without sparsity.
- **Lasso ($L_1$):** Minimizes $\|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 + \alpha \|\mathbf{w}\|_1$. Coordinate descent soft-thresholding sets non-informative weights to identically zero.

In this notebook, we trace the coefficient regularization paths and perform proper cross-validated parameter tuning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

# Generate synthetic multicollinear dataset: 100 samples, 10 features (only 4 informative)
X, y = make_regression(n_samples=150, n_features=10, n_informative=4, noise=8.0, random_state=42)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

### Tracing Regularization Paths Across Alphas
We sweep penalty parameter $\alpha \in [10^{-2}, 10^4]$ on a logarithmic scale.
Notice:
- In Ridge, coefficients shrink asymptotically toward zero but never hit zero.
- In Lasso, coefficients hit zero abruptly at diamond vertex corners, performing automatic feature selection.

In [ ]:
alphas = np.logspace(-2, 4, 100)
ridge_coefs = []
lasso_coefs = []

for a in alphas:
    r = Ridge(alpha=a).fit(X_train, y_train)
    l = Lasso(alpha=a, max_iter=5000).fit(X_train, y_train)
    ridge_coefs.append(r.coef_)
    lasso_coefs.append(l.coef_)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(alphas, ridge_coefs)
plt.xscale('log')
plt.xlabel('Regularization Penalty $\alpha$', fontsize=11)
plt.ylabel('Weights $w_j$', fontsize=11)
plt.title('Ridge ($L_2$) Shrinkage Path (Smooth)', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(alphas, lasso_coefs)
plt.xscale('log')
plt.xlabel('Regularization Penalty $\alpha$', fontsize=11)
plt.ylabel('Weights $w_j$', fontsize=11)
plt.title('Lasso ($L_1$) Sparsity Path (Corners to Zero)', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Hyperparameter Tuning: Selecting Optimal $\alpha$ via Cross-Validation
We use `RidgeCV` and `LassoCV` with 5-fold cross-validation.
- When $\alpha$ is too small: model has high variance and fits collinear noise.
- When $\alpha$ is too large: model has high bias, over-shrinking valid predictive signals.

In [ ]:
ridge_cv = RidgeCV(alphas=np.logspace(-2, 3, 50), cv=5).fit(X_train, y_train)
lasso_cv = LassoCV(alphas=np.logspace(-2, 3, 50), cv=5, max_iter=5000, random_state=42).fit(X_train, y_train)

print(f"Optimal Ridge alpha: {ridge_cv.alpha_:.4f} | Test R2: {ridge_cv.score(X_test, y_test):.4f}")
print(f"Optimal Lasso alpha: {lasso_cv.alpha_:.4f} | Test R2: {lasso_cv.score(X_test, y_test):.4f}")

zero_features = np.sum(lasso_cv.coef_ == 0)
print(f"Features eliminated to zero by Lasso: {zero_features} of {X.shape[1]}")